In [1]:
import mrcfile
import numpy as np
from scipy.ndimage import binary_dilation, generate_binary_structure, binary_opening, binary_erosion, binary_closing
from scipy.ndimage import label
from scipy.ndimage import measurements

In [2]:
base_name = 'pp199'
working_dir = '/media/liushuo/data1/data/fig_demo_2/pp199/synapse_seg/'

In [3]:
def get_tomo(path):
    """
    Load a 3D MRC file as a numpy array.

    Parameters:
    - path: str
        Path to the MRC file.

    Returns:
    - data: ndarray
        The 3D data loaded from the MRC file.
    """
    with mrcfile.open(path) as mrc:
        data = mrc.data
    return data

def save_tomo(data, path, voxel_size=17.14):
    """
    Save a 3D numpy array as an MRC file.

    Parameters:
    - data: ndarray
        The 3D data to save.mrcvesicle
    - voxel_size: float
        The voxel size of the data.
    """
    with mrcfile.new(path, overwrite=True) as mrc:
        data = data.astype(np.int8)
        mrc.set_data(data)
        mrc.voxel_size = voxel_size
        
def binarize_mask(mask: np.ndarray) -> np.ndarray:
    """
    将输入的三维mask数组进行01化处理，原本为0的不变，大于0的置为1。
    
    参数:
    mask (np.ndarray): 输入的三维np数组，代表实例分割的像素级mask。
    
    返回:
    np.ndarray: 处理后的二值化mask数组。
    """
    return np.where(mask > 0, 1, 0)

In [4]:
all_membrane_mask = get_tomo(f'{working_dir}/{base_name}/predictions/{base_name}_MemBrain_seg_v10_alpha.ckpt_segmented.mrc')
vesicle_mask = get_tomo(f'{working_dir}/{base_name}/vesicle/{base_name}_vesicle_label.mrc')

In [5]:
# 对每个mask进行二值化处理
membrane_mask_bin = binarize_mask(all_membrane_mask)
vesicle_mask_bin = binarize_mask(vesicle_mask)

In [6]:
def create_spherical_structure(radius):
    """
    创建一个球形的结构元，用于膨胀操作。

    参数:
    radius (int): 球的半径。

    返回:
    np.ndarray: 球形结构元。
    """
    # 生成球体网格
    size = 2 * radius + 1  # 球形结构元的大小
    z, y, x = np.ogrid[-radius:radius+1, -radius:radius+1, -radius:radius+1]
    distance = x**2 + y**2 + z**2
    # 将距离球心半径以内的点设为1，形成球形结构元
    spherical_structure = distance <= radius**2
    return spherical_structure

# 创建球形结构元
spherical_structure = create_spherical_structure(3)  # 生成一个3D的结构元
# 对合并的mask进行球形膨胀
dilated_vesicle = binary_dilation(vesicle_mask_bin, structure=spherical_structure)

# save_tomo(dilated_mask, '/media/liushuo/data1/data/actin trace/actintomo/20200820/tomo/pp1776/combined_mask.mrc')

In [7]:
vesicle_memb_mask = dilated_vesicle & membrane_mask_bin
save_tomo(vesicle_memb_mask, f'{working_dir}/{base_name}/vesicle/{base_name}_vesicle_memb_label.mrc')